# Flight Delay & Airline Operations Analytics

**Goal:** Perform large-scale operational EDA on 2015 U.S. domestic flights and create SQL- and Power BI-ready outputs.

### Questions
- Which airlines, airports and routes show the highest delay risk?
- When do delays peak by month, weekday and scheduled departure hour?
- Why are flights cancelled?
- How strongly does departure delay translate into arrival delay?
- Which airlines recover the most time after a delayed departure?

In [ ]:
# Optional in Colab / fresh environment:
# %pip install -q pandas numpy matplotlib kagglehub

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Use None for final full-dataset analysis.
SAMPLE_N = None

# Cap only the row-level Power BI export. Aggregate outputs still use all loaded rows.
POWERBI_MAX_ROWS = 1_000_000

RANDOM_STATE = 42

## 1. Load the dataset

In [ ]:
def locate_data():
    manual = DATA_DIR
    expected = [
        manual / "flights.csv",
        manual / "airlines.csv",
        manual / "airports.csv"
    ]
    if all(p.exists() for p in expected):
        return manual

    try:
        import kagglehub
        kaggle_path = Path(kagglehub.dataset_download("usdot/flight-delays"))
        if (kaggle_path / "flights.csv").exists():
            return kaggle_path
    except Exception as exc:
        print("Automatic Kaggle download unavailable:", exc)

    raise FileNotFoundError(
        "Dataset not found. Download https://www.kaggle.com/datasets/usdot/flight-delays "
        "and place flights.csv, airlines.csv and airports.csv inside ./data/"
    )

SOURCE_DIR = locate_data()
print("Using data from:", SOURCE_DIR)

In [ ]:
flight_cols = [
    "YEAR", "MONTH", "DAY", "DAY_OF_WEEK", "AIRLINE", "FLIGHT_NUMBER",
    "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "SCHEDULED_DEPARTURE",
    "DEPARTURE_DELAY", "ARRIVAL_DELAY", "DISTANCE", "DIVERTED", "CANCELLED",
    "CANCELLATION_REASON", "AIR_SYSTEM_DELAY", "SECURITY_DELAY",
    "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY"
]

flights_path = SOURCE_DIR / "flights.csv"
airlines_path = SOURCE_DIR / "airlines.csv"
airports_path = SOURCE_DIR / "airports.csv"

if SAMPLE_N:
    flights = pd.read_csv(
        flights_path, usecols=flight_cols, nrows=SAMPLE_N, low_memory=False
    )
else:
    flights = pd.read_csv(
        flights_path, usecols=flight_cols, low_memory=False
    )

airlines = pd.read_csv(airlines_path)
airports = pd.read_csv(airports_path)

print("Flights:", flights.shape)
print("Airlines:", airlines.shape)
print("Airports:", airports.shape)
display(flights.head())

## 2. Data quality & cleaning

In [ ]:
print("Duplicate rows:", flights.duplicated().sum())

missing = (
    flights.isna().mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_pct")
    .to_frame()
)
display(missing.head(15))

In [ ]:
for col in ["AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]:
    flights[col] = flights[col].astype("string").str.strip()

flights["FLIGHT_DATE"] = pd.to_datetime(
    dict(
        year=flights["YEAR"],
        month=flights["MONTH"],
        day=flights["DAY"]
    ),
    errors="coerce"
)

weekday_map = {
    1: "Monday", 2: "Tuesday", 3: "Wednesday", 4: "Thursday",
    5: "Friday", 6: "Saturday", 7: "Sunday"
}
flights["WEEKDAY_NAME"] = flights["DAY_OF_WEEK"].map(weekday_map)

flights["DEPARTURE_HOUR"] = (
    pd.to_numeric(flights["SCHEDULED_DEPARTURE"], errors="coerce") // 100
).clip(0, 23).astype("Int64")

flights["ROUTE"] = (
    flights["ORIGIN_AIRPORT"] + " → " + flights["DESTINATION_AIRPORT"]
)

cancel_map = {
    "A": "Carrier",
    "B": "Weather",
    "C": "National Air System",
    "D": "Security"
}
flights["CANCELLATION_REASON_LABEL"] = (
    flights["CANCELLATION_REASON"].map(cancel_map)
)
flights.loc[
    flights["CANCELLED"].eq(0),
    "CANCELLATION_REASON_LABEL"
] = "Not Cancelled"

eligible = (
    flights["CANCELLED"].eq(0)
    & flights["DIVERTED"].eq(0)
    & flights["ARRIVAL_DELAY"].notna()
)

flights["IS_ELIGIBLE"] = eligible.astype("int8")

flights["IS_DELAYED"] = (
    eligible & flights["ARRIVAL_DELAY"].ge(15)
).astype("int8")

flights["IS_ON_TIME"] = (
    eligible & flights["ARRIVAL_DELAY"].lt(15)
).astype("int8")

flights["DELAY_RECOVERY"] = (
    flights["DEPARTURE_DELAY"] - flights["ARRIVAL_DELAY"]
)

airline_code_col = (
    "IATA_CODE" if "IATA_CODE" in airlines.columns else airlines.columns[0]
)
airline_name_col = (
    "AIRLINE" if "AIRLINE" in airlines.columns else airlines.columns[1]
)

airline_lookup = (
    airlines.set_index(airline_code_col)[airline_name_col].to_dict()
)

flights["AIRLINE_NAME"] = (
    flights["AIRLINE"].map(airline_lookup).fillna(flights["AIRLINE"])
)

print("Cleaned shape:", flights.shape)
display(flights.head())

## 3. Headline KPIs

In [ ]:
eligible_count = int(eligible.sum())
total_flights = len(flights)
cancelled_flights = int(flights["CANCELLED"].sum())
diverted_flights = int(flights["DIVERTED"].sum())
delayed_flights = int(flights["IS_DELAYED"].sum())
on_time_flights = int(flights["IS_ON_TIME"].sum())

corr_base = flights.loc[
    eligible & flights["DEPARTURE_DELAY"].notna(),
    ["DEPARTURE_DELAY", "ARRIVAL_DELAY"]
].dropna()

corr_value = (
    corr_base.corr().iloc[0, 1]
    if len(corr_base) > 1 else np.nan
)

kpi = pd.DataFrame({
    "metric": [
        "Total Flights",
        "Eligible Completed Flights",
        "On-Time Flights",
        "Delayed Flights (15+ min)",
        "Cancelled Flights",
        "Diverted Flights",
        "On-Time %",
        "Delay Rate %",
        "Cancellation %",
        "Avg Arrival Delay (min)",
        "Avg Departure Delay (min)",
        "Departure-Arrival Delay Correlation"
    ],
    "value": [
        total_flights,
        eligible_count,
        on_time_flights,
        delayed_flights,
        cancelled_flights,
        diverted_flights,
        100 * on_time_flights / eligible_count if eligible_count else np.nan,
        100 * delayed_flights / eligible_count if eligible_count else np.nan,
        100 * cancelled_flights / total_flights if total_flights else np.nan,
        flights.loc[eligible, "ARRIVAL_DELAY"].mean(),
        flights.loc[
            flights["CANCELLED"].eq(0), "DEPARTURE_DELAY"
        ].mean(),
        corr_value
    ]
})

display(kpi)
kpi.to_csv(OUTPUT_DIR / "kpi_summary.csv", index=False)

## 4. Airline performance

In [ ]:
def performance_table(df, group_col, min_flights=1):
    temp = (
        df.groupby(group_col, dropna=False)
        .agg(
            total_flights=("FLIGHT_NUMBER", "size"),
            eligible_flights=("IS_ELIGIBLE", "sum"),
            delayed_flights=("IS_DELAYED", "sum"),
            cancelled_flights=("CANCELLED", "sum"),
            avg_arrival_delay=("ARRIVAL_DELAY", "mean"),
            avg_departure_delay=("DEPARTURE_DELAY", "mean")
        )
        .reset_index()
    )

    temp["delay_rate_pct"] = (
        100 * temp["delayed_flights"] /
        temp["eligible_flights"].replace(0, np.nan)
    )
    temp["cancellation_rate_pct"] = (
        100 * temp["cancelled_flights"] /
        temp["total_flights"].replace(0, np.nan)
    )
    return temp[temp["total_flights"] >= min_flights].copy()

airline_perf = performance_table(
    flights, "AIRLINE_NAME", min_flights=10_000
).sort_values("delay_rate_pct", ascending=False)

display(airline_perf.head(15))
airline_perf.to_csv(
    OUTPUT_DIR / "airline_performance.csv", index=False
)

plot_df = airline_perf.head(12).sort_values("delay_rate_pct")
plt.figure(figsize=(10, 6))
plt.barh(plot_df["AIRLINE_NAME"], plot_df["delay_rate_pct"])
plt.xlabel("Arrival delay rate (%)")
plt.ylabel("")
plt.title("Airlines with Highest 15+ Minute Arrival Delay Rate")
plt.tight_layout()
plt.show()

## 5. Airport performance

In [ ]:
airport_perf = (
    flights.groupby("ORIGIN_AIRPORT", dropna=False)
    .agg(
        departures=("FLIGHT_NUMBER", "size"),
        eligible_flights=("IS_ELIGIBLE", "sum"),
        avg_departure_delay=("DEPARTURE_DELAY", "mean"),
        delayed_flights=("IS_DELAYED", "sum"),
        cancelled_flights=("CANCELLED", "sum")
    )
    .reset_index()
)

airport_perf["delay_rate_pct"] = (
    100 * airport_perf["delayed_flights"] /
    airport_perf["eligible_flights"].replace(0, np.nan)
)
airport_perf["cancellation_rate_pct"] = (
    100 * airport_perf["cancelled_flights"] /
    airport_perf["departures"]
)

airport_perf = airport_perf[
    airport_perf["departures"] >= 10_000
].sort_values("avg_departure_delay", ascending=False)

display(airport_perf.head(20))
airport_perf.to_csv(
    OUTPUT_DIR / "airport_performance.csv", index=False
)

plot_df = airport_perf.head(15).sort_values("avg_departure_delay")
plt.figure(figsize=(10, 6))
plt.barh(
    plot_df["ORIGIN_AIRPORT"].astype(str),
    plot_df["avg_departure_delay"]
)
plt.xlabel("Average departure delay (minutes)")
plt.ylabel("Origin airport")
plt.title("Origin Airports with Highest Average Departure Delay")
plt.tight_layout()
plt.show()

## 6. Route operational risk

In [ ]:
route_perf = (
    flights.groupby("ROUTE", dropna=False)
    .agg(
        total_flights=("FLIGHT_NUMBER", "size"),
        eligible_flights=("IS_ELIGIBLE", "sum"),
        delayed_flights=("IS_DELAYED", "sum"),
        cancelled_flights=("CANCELLED", "sum"),
        avg_arrival_delay=("ARRIVAL_DELAY", "mean")
    )
    .reset_index()
)

route_perf["delay_rate_pct"] = (
    100 * route_perf["delayed_flights"] /
    route_perf["eligible_flights"].replace(0, np.nan)
)
route_perf["cancellation_rate_pct"] = (
    100 * route_perf["cancelled_flights"] /
    route_perf["total_flights"].replace(0, np.nan)
)

route_perf = route_perf[
    route_perf["total_flights"] >= 1_000
].copy()

route_perf["operational_risk_score"] = (
    0.70 * route_perf["delay_rate_pct"] +
    0.30 * route_perf["cancellation_rate_pct"]
)

route_perf = route_perf.sort_values(
    "operational_risk_score", ascending=False
)

display(route_perf.head(20))
route_perf.to_csv(
    OUTPUT_DIR / "route_performance.csv", index=False
)

plot_df = route_perf.head(15).sort_values("operational_risk_score")
plt.figure(figsize=(10, 7))
plt.barh(
    plot_df["ROUTE"].astype(str),
    plot_df["operational_risk_score"]
)
plt.xlabel("Operational risk score")
plt.ylabel("Route")
plt.title("Highest-Risk High-Volume Routes")
plt.tight_layout()
plt.show()

## 7. Monthly, weekday & hourly trends

In [ ]:
monthly = (
    flights.groupby("MONTH")
    .agg(
        total_flights=("FLIGHT_NUMBER", "size"),
        eligible_flights=("IS_ELIGIBLE", "sum"),
        delayed_flights=("IS_DELAYED", "sum"),
        cancelled_flights=("CANCELLED", "sum"),
        avg_arrival_delay=("ARRIVAL_DELAY", "mean")
    )
    .reset_index()
)

monthly["delay_rate_pct"] = (
    100 * monthly["delayed_flights"] / monthly["eligible_flights"].replace(0, np.nan)
)
monthly["cancellation_rate_pct"] = (
    100 * monthly["cancelled_flights"] / monthly["total_flights"]
)
monthly.to_csv(OUTPUT_DIR / "monthly_trends.csv", index=False)

display(monthly)

plt.figure(figsize=(10, 5))
plt.plot(
    monthly["MONTH"],
    monthly["delay_rate_pct"],
    marker="o"
)
plt.xticks(range(1, 13))
plt.xlabel("Month")
plt.ylabel("Delay rate (%)")
plt.title("Monthly Flight Delay Trend")
plt.tight_layout()
plt.show()

In [ ]:
weekday = (
    flights.groupby(["DAY_OF_WEEK", "WEEKDAY_NAME"])
    .agg(
        total_flights=("FLIGHT_NUMBER", "size"),
        eligible_flights=("IS_ELIGIBLE", "sum"),
        delayed_flights=("IS_DELAYED", "sum"),
        cancelled_flights=("CANCELLED", "sum"),
        avg_arrival_delay=("ARRIVAL_DELAY", "mean")
    )
    .reset_index()
    .sort_values("DAY_OF_WEEK")
)

weekday["delay_rate_pct"] = (
    100 * weekday["delayed_flights"] / weekday["eligible_flights"].replace(0, np.nan)
)
weekday["cancellation_rate_pct"] = (
    100 * weekday["cancelled_flights"] / weekday["total_flights"]
)
weekday.to_csv(OUTPUT_DIR / "weekday_trends.csv", index=False)

display(weekday)

plt.figure(figsize=(10, 5))
plt.bar(weekday["WEEKDAY_NAME"], weekday["delay_rate_pct"])
plt.xticks(rotation=30)
plt.ylabel("Delay rate (%)")
plt.title("Flight Delay Rate by Day of Week")
plt.tight_layout()
plt.show()

In [ ]:
hourly = (
    flights.dropna(subset=["DEPARTURE_HOUR"])
    .groupby("DEPARTURE_HOUR")
    .agg(
        total_flights=("FLIGHT_NUMBER", "size"),
        eligible_flights=("IS_ELIGIBLE", "sum"),
        delayed_flights=("IS_DELAYED", "sum"),
        avg_departure_delay=("DEPARTURE_DELAY", "mean")
    )
    .reset_index()
)

hourly["delay_rate_pct"] = (
    100 * hourly["delayed_flights"] / hourly["eligible_flights"].replace(0, np.nan)
)
hourly.to_csv(OUTPUT_DIR / "hourly_trends.csv", index=False)

display(hourly)

plt.figure(figsize=(10, 5))
plt.plot(
    hourly["DEPARTURE_HOUR"],
    hourly["delay_rate_pct"],
    marker="o"
)
plt.xticks(range(0, 24))
plt.xlabel("Scheduled departure hour")
plt.ylabel("Delay rate (%)")
plt.title("Delay Rate by Scheduled Departure Hour")
plt.tight_layout()
plt.show()

## 8. Cancellation analysis

In [ ]:
cancel_reasons = (
    flights.loc[
        flights["CANCELLED"].eq(1),
        "CANCELLATION_REASON_LABEL"
    ]
    .fillna("Unknown")
    .value_counts()
    .rename_axis("cancellation_reason")
    .reset_index(name="cancelled_flights")
)

cancel_reasons["share_pct"] = (
    100 * cancel_reasons["cancelled_flights"] /
    cancel_reasons["cancelled_flights"].sum()
)

display(cancel_reasons)
cancel_reasons.to_csv(
    OUTPUT_DIR / "cancellation_reasons.csv", index=False
)

plt.figure(figsize=(9, 5))
plt.bar(
    cancel_reasons["cancellation_reason"],
    cancel_reasons["cancelled_flights"]
)
plt.ylabel("Cancelled flights")
plt.title("Cancellation Reasons")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 9. Departure vs arrival delay correlation

In [ ]:
corr_df = flights.loc[
    eligible & flights["DEPARTURE_DELAY"].notna(),
    ["DEPARTURE_DELAY", "ARRIVAL_DELAY"]
].dropna()

corr = corr_df.corr().iloc[0, 1]
print(f"Pearson correlation: {corr:.3f}")

sample_plot = corr_df.sample(
    min(50_000, len(corr_df)),
    random_state=RANDOM_STATE
)

plt.figure(figsize=(8, 6))
plt.hexbin(
    sample_plot["DEPARTURE_DELAY"],
    sample_plot["ARRIVAL_DELAY"],
    gridsize=60,
    mincnt=1
)
plt.xlabel("Departure delay (minutes)")
plt.ylabel("Arrival delay (minutes)")
plt.title(f"Departure vs Arrival Delay | r = {corr:.3f}")
plt.xlim(-30, 240)
plt.ylim(-60, 240)
plt.tight_layout()
plt.show()

## 10. Delay-cause analysis

In [ ]:
cause_cols = {
    "AIR_SYSTEM_DELAY": "National Air System",
    "SECURITY_DELAY": "Security",
    "AIRLINE_DELAY": "Airline",
    "LATE_AIRCRAFT_DELAY": "Late Aircraft",
    "WEATHER_DELAY": "Weather"
}

delay_causes = (
    flights[list(cause_cols)]
    .sum(numeric_only=True)
    .rename(index=cause_cols)
    .sort_values(ascending=False)
    .rename("delay_minutes")
    .reset_index()
    .rename(columns={"index": "delay_cause"})
)

delay_causes["share_pct"] = (
    100 * delay_causes["delay_minutes"] /
    delay_causes["delay_minutes"].sum()
)

display(delay_causes)
delay_causes.to_csv(
    OUTPUT_DIR / "delay_causes.csv", index=False
)

plt.figure(figsize=(9, 5))
plt.bar(
    delay_causes["delay_cause"],
    delay_causes["delay_minutes"] / 1_000_000
)
plt.ylabel("Delay minutes (millions)")
plt.title("Total Delay Minutes by Cause")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 11. Airline delay recovery

In [ ]:
recovery_base = flights[
    flights["DEPARTURE_DELAY"].gt(0)
    & flights["CANCELLED"].eq(0)
    & flights["DIVERTED"].eq(0)
    & flights["ARRIVAL_DELAY"].notna()
].copy()

recovery = (
    recovery_base.groupby("AIRLINE_NAME")
    .agg(
        delayed_departures=("FLIGHT_NUMBER", "size"),
        avg_departure_delay=("DEPARTURE_DELAY", "mean"),
        avg_arrival_delay=("ARRIVAL_DELAY", "mean"),
        avg_minutes_recovered=("DELAY_RECOVERY", "mean")
    )
    .reset_index()
)

recovery = recovery[
    recovery["delayed_departures"] >= 5_000
].sort_values("avg_minutes_recovered", ascending=False)

display(recovery)
recovery.to_csv(
    OUTPUT_DIR / "airline_delay_recovery.csv", index=False
)

plot_df = recovery.head(12).sort_values("avg_minutes_recovered")
plt.figure(figsize=(10, 6))
plt.barh(
    plot_df["AIRLINE_NAME"],
    plot_df["avg_minutes_recovered"]
)
plt.xlabel("Average minutes recovered")
plt.ylabel("")
plt.title("Airlines Recovering Most Time After Delayed Departures")
plt.tight_layout()
plt.show()

## 12. Automatically generate portfolio insights

In [ ]:
def first_row(df, sort_col):
    if df.empty:
        return None
    return df.sort_values(
        sort_col, ascending=False
    ).iloc[0]

worst_airline = first_row(airline_perf, "delay_rate_pct")
worst_airport = first_row(airport_perf, "avg_departure_delay")
worst_route = first_row(route_perf, "operational_risk_score")
worst_month = first_row(monthly, "delay_rate_pct")
worst_hour = first_row(hourly, "delay_rate_pct")
best_recovery = first_row(recovery, "avg_minutes_recovered")

insights = []

if worst_airline is not None:
    insights.append(
        f"{worst_airline['AIRLINE_NAME']} had the highest delay rate among "
        f"airlines meeting the volume threshold: "
        f"{worst_airline['delay_rate_pct']:.1f}%."
    )

if worst_airport is not None:
    insights.append(
        f"{worst_airport['ORIGIN_AIRPORT']} had the highest average departure "
        f"delay among high-volume origin airports: "
        f"{worst_airport['avg_departure_delay']:.1f} minutes."
    )

if worst_route is not None:
    insights.append(
        f"{worst_route['ROUTE']} ranked highest on the defined high-volume route "
        f"risk score ({worst_route['operational_risk_score']:.1f})."
    )

if worst_month is not None:
    insights.append(
        f"Month {int(worst_month['MONTH'])} had the highest delay rate: "
        f"{worst_month['delay_rate_pct']:.1f}%."
    )

if worst_hour is not None:
    insights.append(
        f"Flights scheduled around "
        f"{int(worst_hour['DEPARTURE_HOUR']):02d}:00 had the highest hourly "
        f"delay rate: {worst_hour['delay_rate_pct']:.1f}%."
    )

if best_recovery is not None:
    insights.append(
        f"{best_recovery['AIRLINE_NAME']} recovered the most time on average "
        f"after delayed departures: "
        f"{best_recovery['avg_minutes_recovered']:.1f} minutes."
    )

print("\n".join(f"• {x}" for x in insights))

(OUTPUT_DIR / "portfolio_insights.txt").write_text(
    "\n".join(f"- {x}" for x in insights),
    encoding="utf-8"
)

## 13. Export Power BI / PostgreSQL-ready row-level data

In [ ]:
export_cols = [
    "FLIGHT_DATE", "YEAR", "MONTH", "DAY", "DAY_OF_WEEK",
    "WEEKDAY_NAME", "AIRLINE", "AIRLINE_NAME", "FLIGHT_NUMBER",
    "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "ROUTE",
    "SCHEDULED_DEPARTURE", "DEPARTURE_HOUR",
    "DEPARTURE_DELAY", "ARRIVAL_DELAY", "DISTANCE",
    "DIVERTED", "CANCELLED", "CANCELLATION_REASON_LABEL",
    "AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY",
    "IS_ELIGIBLE", "IS_DELAYED", "IS_ON_TIME", "DELAY_RECOVERY"
]

powerbi_df = flights[export_cols].copy()

if len(powerbi_df) > POWERBI_MAX_ROWS:
    powerbi_df = powerbi_df.sample(
        POWERBI_MAX_ROWS,
        random_state=RANDOM_STATE
    )

powerbi_df.columns = [
    c.lower() for c in powerbi_df.columns
]

powerbi_df.to_csv(
    OUTPUT_DIR / "flights_powerbi.csv",
    index=False
)

print(
    f"Exported {len(powerbi_df):,} rows to "
    "outputs/flights_powerbi.csv"
)

## 14. Final portfolio checklist

Before putting this project on your CV:

1. Set `SAMPLE_N = None`.
2. Run the notebook on the full dataset.
3. Check `outputs/portfolio_insights.txt`.
4. Build the Power BI dashboard from the exported files.
5. Use only verified, quantified findings in CV bullets.
6. Never claim a metric generated only from a development sample.